In [4]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2

model = YOLO('yolov8n.pt')

print("YOLOv8 Model Loaded!")

YOLOv8 Model Loaded!


In [2]:
results = model("traffic.jpg")
results[0].show()


image 1/1 C:\Users\Nikhil\traffic.jpg: 448x640 12 persons, 3 cars, 3 motorcycles, 3 buss, 1 truck, 1 traffic light, 203.5ms
Speed: 16.5ms preprocess, 203.5ms inference, 6.2ms postprocess per image at shape (1, 3, 448, 640)


In [3]:
from roboflow import Roboflow
rf = Roboflow(api_key="qjRTVY2K65aWyPpXEG61")
project = rf.workspace("indian-currency-dtw0l").project("indian-currency-wpudx")
version = project.version(1)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...


In [4]:
with open(f"{dataset.location}/data.yaml", 'r') as f:
    print(f.read())

names:
- '10'
- '100'
- '20'
- '200'
- '2000'
- '50'
- '500'
nc: 7
roboflow:
  license: CC BY 4.0
  project: indian-currency-wpudx
  url: https://universe.roboflow.com/indian-currency-dtw0l/indian-currency-wpudx/dataset/1
  version: 1
  workspace: indian-currency-dtw0l
test: ../test/images
train: ../train/images
val: ../valid/images



In [10]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=5,
    imgsz=416,
    batch=8,
    name='currency_detector'
)

print(" Fine-tuning Complete!")

Ultralytics 8.4.67  Python-3.13.9 torch-2.12.0+cpu CPU (12th Gen Intel Core i5-12450HX)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Nikhil\Indian-Currency-1/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=currency_detector-3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, ov

In [3]:
from ultralytics import YOLO

trained_model = YOLO(r'C:\Users\Nikhil\runs\detect\currency_detector-3\weights\best.pt')

print("Fine-tuned Currency Model Loaded!")

Fine-tuned Currency Model Loaded!


In [5]:
results = trained_model("currency_photo.jpg")
results[0].show()


image 1/1 C:\Users\Nikhil\currency_photo.jpg: 192x416 1 500, 388.4ms
Speed: 122.0ms preprocess, 388.4ms inference, 30.9ms postprocess per image at shape (1, 3, 192, 416)


In [6]:
original_model = YOLO('yolov8n.pt')
results1 = original_model("currency_photo.jpg")
results1[0].show()
print("BEFORE Fine-tuning: Generic detection")

results2 = trained_model("currency_photo.jpg")
results2[0].show()
print("AFTER Fine-tuning: Currency-specific detection!")


image 1/1 C:\Users\Nikhil\currency_photo.jpg: 320x640 1 person, 415.2ms
Speed: 9.3ms preprocess, 415.2ms inference, 7.6ms postprocess per image at shape (1, 3, 320, 640)
BEFORE Fine-tuning: Generic detection

image 1/1 C:\Users\Nikhil\currency_photo.jpg: 192x416 1 500, 217.6ms
Speed: 4.3ms preprocess, 217.6ms inference, 3.9ms postprocess per image at shape (1, 3, 192, 416)
AFTER Fine-tuning: Currency-specific detection!


In [1]:
from ultralytics import YOLO
from ipywidgets import FileUpload, Button, Output
from IPython.display import display
import io
from PIL import Image
import matplotlib.pyplot as plt

trained_model = YOLO(r'C:\Users\Nikhil\runs\detect\currency_detector-3\weights\best.pt')
print("Fine-tuned Model Loaded!")

upload = FileUpload(accept='image/*', multiple=False)
btn = Button(description="Detect Currency!", button_style='success')
out = Output()
display(upload, btn, out)

def detect_currency(b):
    out.clear_output()
    with out:
        if len(upload.value) == 0:
            print("Upload image first")
            return
        
        uploaded_file = list(upload.value)[0]
        img_bytes = uploaded_file['content']
        
        img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        img.save('temp_currency.jpg')
        
        results = trained_model('temp_currency.jpg')
        
        result_img = results[0].plot()  
        result_img = result_img[:, :, ::-1]  
        
        plt.figure(figsize=(8, 6))
        plt.imshow(result_img)
        plt.axis('off')
        plt.show()
    
        for box in results[0].boxes:
            cls_name = trained_model.names[int(box.cls)]
            conf = float(box.conf) * 100
            print(f"Detected: ₹{cls_name}  |  Confidence: {conf:.1f}%")

btn.on_click(detect_currency)

Fine-tuned Model Loaded!


FileUpload(value=(), accept='image/*', description='Upload')

Button(button_style='success', description='Detect Currency!', style=ButtonStyle())

Output()